# 7.19 回答能寫多長，和前文能放多少，有何不同？


你先告訴助手「店名是小小書店」，它回「好」。下一輪問「現在只答店名」，希望它寫出「小小書店」就結束。這次回答很短，助手卻仍要讀到前面的店名、誰說了哪句話，以及現在輪到它回答。只數最後這個問題，會漏掉真正送進模型的前文。

我們把對話排成一串，用一個只有 16 個位置的紙上例子看清楚。這裡特別指定一份玩具字表：〈user〉是使用者標記，〈assistant〉是助手標記，〈eos〉就是 EOS，表示這一段結束；每個標記各算一個 token。文字片段「店名是小小書店。」「好。」「現在」「只答」「店名。」「小小」「書店」也各自指定為一個 token。這是為了容易數位置而訂的規則，沒有說一般模型把一句話算一個 token，也沒有說每個中文字都算一個 token。

照這份字表，已提供的歷史和目前的問題排成下面的序列。表中每個由斜線分開的項目佔一個位置，斜線只用來顯示分隔，不送入模型。

| 這段訊息 | 依序放入的token | 位置數 |
| --- | --- | ---: |
| 先前使用者說店名 | 〈user〉／店名是小小書店。／〈eos〉 | 3 |
| 先前助手確認 | 〈assistant〉／好。／〈eos〉 | 3 |
| 現在的要求 | 〈user〉／現在／只答／店名。／〈eos〉 | 5 |
| 輪到助手回答 | 〈assistant〉 | 1 |

全部輸入佔 12 個位置。最後的助手標記已經在輸入裡，下一步才開始生成答案；歷史裡的結束標記也佔位置，但不會讓新的回答立即停止。把對話依角色與文字排成這種輸入序列，叫作序列化。實際使用別的模型時，要用它配套的對話格式與 tokenizer 數這份序列，不能直接沿用這裡的 12。

圖中 U、A、E 分別代表三種標記，a 到 g 代表上面指定的文字片段。先看第 1 到 12 個位置：它們都是已送入的前文。第 13 到 16 個位置才是預留給新回答的空間；第 13 到 15 個位置填入的 f、g、E 是手寫的示意回答，第 16 個位置沒有用到。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/new-7.19-sequence-budget.svg")))

這份生成器保留完整歷史，並規定「前文加上新生成的 token」最多 16 個位置；這個總上限就是上下文視窗（context window）的長度，記為 T。前文的長度記為 P，本例 P＝12。另有一個設定 G，只限制這次最多新增幾個 token，常見名稱是 max_new_tokens。G 不包含已送入的 12 個位置，也不會把 T 改大，所以必須滿足 P＋G≤T。下面用手寫的候選代替模型每一步選出的 token，試三種 G：


In [ ]:
T = 16
prompt = ["〈user〉", "店名是小小書店。", "〈eos〉", "〈assistant〉", "好。", "〈eos〉"]
prompt += ["〈user〉", "現在", "只答", "店名。", "〈eos〉", "〈assistant〉"]
P = len(prompt)
print("前文長度P", P, "還能生成的位置T-P", T - P)
candidates = ["小小", "書店", "〈eos〉", "謝謝"]
for G in [4, 2, 5]:
    if P + G > T:
        print(f"G={G}：P+G={P + G} 超過 T={T}，拒絕這個設定")
        continue
    reply = []
    for token in candidates[:G]:
        reply.append(token)
        if token == "〈eos〉":
            break
    reason = "模型選了EOS" if reply[-1] == "〈eos〉" else "用完G"
    print(f"G={G}：新增{len(reply)}個token {reply}，停止原因：{reason}")

第一行印出 P＝12，剩下 T−P＝4 個位置可以生成。G＝4 時，候選在第三個 token 選了〈eos〉，`break` 提早結束，只新增 3 個 token，最後一個預留位置沒有用到。EOS 也算在新增的數量裡：它雖然不顯示成回答文字，仍是模型選出的一個 token。沒用到的位置不是補進輸入的 PAD，只是空間還沒用完。G＝2 時只能留下「小小／書店」，讀者已看得到完整店名，卻還沒有生成〈eos〉：這次是預算用完而截停。G＝5 時 P＋G＝17，超過 T＝16，程式先拒絕這個設定，而不是假裝多出的空間存在；`continue` 跳過這一輪剩下的程式，直接試下一個 G。

反過來，若第一步就選〈eos〉，雖然提早停止，店名仍沒有回答。回答長度、停止原因與內容是否完成，各自告訴我們不同的事。

練習保持前文不變，把 candidates 改成 `["小小", "〈eos〉"]`。先預測 G＝4 時新增幾個 token、有沒有超過 G、是否完成「只答店名」，再執行核對：應得到 2 個、沒有超過、沒有完成。空間足夠、主動結束，都不能替少掉的「書店」補上證據。

<details>
<summary>補充：長度設定的來源與這個算例的範圍</summary>

Hugging Face Transformers v4.57.1 的 [GenerationConfig 文件](https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/generation/configuration_utils.py)把 max_new_tokens 定義為不含 prompt 的新生成上限；vLLM v0.8.5 的 [ModelConfig 文件](https://github.com/vllm-project/vllm/blob/v0.8.5/vllm/config.py)把 max_model_len 定義為 prompt 與 output 的總長度。本節採用保留完整歷史、共用一個總上限的簡化例子；滑動視窗或其他模型介面，要另看它們的長度規定。

實際核對正常結束時，還要保存原始 token 與停止原因，分清模型選了 EOS、程式強制加上 EOS 和其他外部停止條件，不能只看畫面末尾。回答目標與 EOS 的監督位置沿用 [7.3](https://birdhackor.github.io/tiny-perceptron-vlm/7.3.html)、[7.4](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html)、[7.9](https://birdhackor.github.io/tiny-perceptron-vlm/7.9.html) 的安排。

</details>
